# One Haut Encoded — Exploratory Data Analysis

H&M Personalized Fashion Recommendations dataset.

**Goal:** Understand data distributions, sparsity, and quality before building recommendation models.

In [ ]:
# AI-assisted (Claude Code, claude.ai) -- https://claude.ai
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

## 1. Load Data

In [ ]:
articles = pd.read_csv(DATA_DIR / "articles.csv", dtype={"article_id": str})
customers = pd.read_csv(DATA_DIR / "customers.csv")
transactions = pd.read_csv(DATA_DIR / "transactions_train.csv", dtype={"article_id": str})
transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])

print(f"Articles:     {articles.shape}")
print(f"Customers:    {customers.shape}")
print(f"Transactions: {transactions.shape}")

In [ ]:
articles.head()

In [ ]:
customers.head()

In [ ]:
transactions.head()

## 2. Missing Data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, df) in zip(axes, [("Articles", articles), ("Customers", customers), ("Transactions", transactions)]):
    missing = df.isnull().mean().sort_values(ascending=False)
    missing = missing[missing > 0]
    if len(missing) == 0:
        ax.text(0.5, 0.5, "No missing data", ha="center", va="center", fontsize=14)
    else:
        missing.plot.barh(ax=ax)
    ax.set_title(f"{name} — Missing Rate")
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

## 3. Transaction Patterns

In [ ]:
# Purchases per customer
purchases_per_customer = transactions.groupby("customer_id").size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

purchases_per_customer.clip(upper=100).hist(bins=50, ax=axes[0], edgecolor="black")
axes[0].set_title("Purchases per Customer (clipped at 100)")
axes[0].set_xlabel("Number of purchases")
axes[0].set_ylabel("Number of customers")

# Purchases per article
purchases_per_article = transactions.groupby("article_id").size()

purchases_per_article.clip(upper=500).hist(bins=50, ax=axes[1], edgecolor="black")
axes[1].set_title("Purchases per Article (clipped at 500)")
axes[1].set_xlabel("Number of purchases")
axes[1].set_ylabel("Number of articles")

plt.tight_layout()
plt.show()

print(f"Purchases per customer — median: {purchases_per_customer.median():.0f}, "
      f"mean: {purchases_per_customer.mean():.1f}, max: {purchases_per_customer.max()}")
print(f"Purchases per article — median: {purchases_per_article.median():.0f}, "
      f"mean: {purchases_per_article.mean():.1f}, max: {purchases_per_article.max()}")

In [ ]:
# Purchases over time
daily = transactions.groupby(transactions["t_dat"].dt.to_period("W")).size()
daily.index = daily.index.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
daily.plot(ax=ax)
ax.set_title("Weekly Transaction Volume")
ax.set_xlabel("Date")
ax.set_ylabel("Transactions")
plt.tight_layout()
plt.show()

In [ ]:
# Price distribution
fig, ax = plt.subplots(figsize=(10, 5))
transactions["price"].clip(upper=transactions["price"].quantile(0.99)).hist(bins=50, ax=ax, edgecolor="black")
ax.set_title("Price Distribution (clipped at 99th percentile)")
ax.set_xlabel("Price")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

print(f"Price — median: {transactions['price'].median():.4f}, "
      f"mean: {transactions['price'].mean():.4f}, "
      f"min: {transactions['price'].min():.4f}, "
      f"max: {transactions['price'].max():.4f}")

## 4. Customer Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution
customers["age"].dropna().clip(upper=100).hist(bins=40, ax=axes[0], edgecolor="black")
axes[0].set_title("Customer Age Distribution")
axes[0].set_xlabel("Age")

# Club member status
customers["club_member_status"].value_counts().plot.bar(ax=axes[1], edgecolor="black")
axes[1].set_title("Club Member Status")
axes[1].tick_params(axis="x", rotation=0)

# Fashion news frequency
customers["fashion_news_frequency"].value_counts().plot.bar(ax=axes[2], edgecolor="black")
axes[2].set_title("Fashion News Frequency")
axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 5. Article Metadata

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top 20 product types
articles["product_type_name"].value_counts().head(20).plot.barh(ax=axes[0, 0])
axes[0, 0].set_title("Top 20 Product Types")
axes[0, 0].invert_yaxis()

# Department
articles["department_name"].value_counts().head(15).plot.barh(ax=axes[0, 1])
axes[0, 1].set_title("Top 15 Departments")
axes[0, 1].invert_yaxis()

# Garment group
articles["garment_group_name"].value_counts().plot.barh(ax=axes[1, 0])
axes[1, 0].set_title("Garment Groups")
axes[1, 0].invert_yaxis()

# Colour group
articles["colour_group_name"].value_counts().head(15).plot.barh(ax=axes[1, 1])
axes[1, 1].set_title("Top 15 Colours")
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Description word cloud
from wordcloud import WordCloud

text = " ".join(articles["detail_desc"].dropna().astype(str))
wc = WordCloud(width=800, height=400, background_color="white", max_words=100).generate(text)

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Article Description Word Cloud")
plt.tight_layout()
plt.show()

## 6. Interaction Matrix Sparsity

In [ ]:
n_customers = transactions["customer_id"].nunique()
n_articles = transactions["article_id"].nunique()
n_interactions = transactions.groupby(["customer_id", "article_id"]).ngroups
n_possible = n_customers * n_articles
sparsity = 1 - n_interactions / n_possible

print(f"Customers:       {n_customers:>12,}")
print(f"Articles:        {n_articles:>12,}")
print(f"Possible pairs:  {n_possible:>12,}")
print(f"Actual pairs:    {n_interactions:>12,}")
print(f"Sparsity:        {sparsity:>12.6%}")
print(f"\nThis extreme sparsity motivates collaborative filtering and content-based approaches.")

## 7. Sample Images

In [ ]:
from PIL import Image
import os

# Sample 4 product types, 3 images each
top_types = articles["product_type_name"].value_counts().head(4).index
image_dir = DATA_DIR / "images"

fig, axes = plt.subplots(4, 3, figsize=(12, 16))

for i, ptype in enumerate(top_types):
    type_articles = articles[articles["product_type_name"] == ptype]["article_id"].values[:3]
    for j, aid in enumerate(type_articles):
        img_path = image_dir / f"0{aid[:3]}" / f"0{aid}.jpg"
        if img_path.exists():
            img = Image.open(img_path)
            axes[i, j].imshow(img)
        else:
            axes[i, j].text(0.5, 0.5, "Missing", ha="center", va="center")
        axes[i, j].set_title(f"{ptype}\n{aid}", fontsize=9)
        axes[i, j].axis("off")

plt.suptitle("Sample Images by Product Type", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Key Takeaways

Fill in after running:

1. **Sparsity:** _____
2. **Long-tail:** _____
3. **Seasonality:** _____
4. **Best metadata coverage:** _____
5. **Image quality:** _____